In [1]:
import os
from collections import defaultdict
from typing import Dict, Set

import regex

In [2]:
special_tokens = ['<|endoftext|>']
vocab : Dict[int , bytes] = {i : bytes([i]) for i in range(256)}
cur_token_id = 256
words_set : Set[bytes] = set(vocab.values())
for special_token in special_tokens:
    sp_token_bytes = special_token.encode('utf-8')
    if sp_token_bytes not in words_set:
        words_set.add(sp_token_bytes)
        vocab[cur_token_id] = sp_token_bytes
        cur_token_id += 1
print(vocab)

{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 87: b'W', 88: b'X', 89: b'Y', 90: b'Z', 91: b'[',

In [ ]:
# word_counts = defaultdict(int)
# pair_counts = defaultdict(int) 
from collections import Counter
import multiprocessing as mp
corpus = ["""u don't have to be scared of the loud dog, I'll protect you". The mole felt so safe with the little girl. She was very kind and the mole soon came to trust her. He leaned against her and she kept him safe. The mole had found his best friend.
<|endoftext|>
Once upon a time, in a warm and sunny place, there was a big pit. A little boy named Tom liked to play near the pit. One day, Tom lost his red ball. He was very sad.
Tom asked his friend, Sam, to help him search for the ball. They looked high and low, but they could not find the ball. Tom said, "I think my ball fell into the pit."
Sam and Tom went close to the pit. They were scared, but they wanted to find the red ball. They looked into the pit, but it was too dark to see. Tom said, "We must go in and search for my ball."
They went into the pit to search. It was dark and scary. They could not find the ball. They tried to get out, but the pit was too deep. Tom and Sam were stuck in the pit. They called for help, but no one could hear them. They were sad and scared, and they never got out of the pit.
<|endoftext|>
Once upon a time, there was a messy giant named Bob. Bob lived in a big forest with many trees. Bob was always making a mess because he was so big and clumsy. One day, Bob decided he wanted to sell his big rocks to the people in the town.
Bob walked to the town with a big bag of rocks. He met a little girl named Sue. Sue looked at the rocks and said, "These rocks are very messy, but I can help you clean them." Bob was happy and thanked Sue. They cleaned the rocks together and made them shiny.
After cleaning the rocks, Bob an""", 
"""u don't have to be scared of the loud dog, I'll protect you". The mole felt so safe with the little girl. She was very kind and the mole soon came to trust her. He leaned against her and she kept him safe. The mole had found his best friend.
<|endoftext|>
Once upon a time, in a warm and sunny place, there was a big pit. A little boy named Tom liked to play near the pit. One day, Tom lost his red ball. He was very sad.
Tom asked his friend, Sam, to help him search for the ball. They looked high and low, but they could not find the ball. Tom said, "I think my ball fell into the pit."
Sam and Tom went close to the pit. They were scared, but they wanted to find the red ball. They looked into the pit, but it was too dark to see. Tom said, "We must go in and search for my ball."
They went into the pit to search. It was dark and scary. They could not find the ball. They tried to get out, but the pit was too deep. Tom and Sam were stuck in the pit. They called for help, but no one could hear them. They were sad and scared, and they never got out of the pit.
<|endoftext|>
Once upon a time, there was a messy giant named Bob. Bob lived in a big forest with many trees. Bob was always making a mess because he was so big and clumsy. One day, Bob decided he wanted to sell his big rocks to the people in the town.
Bob walked to the town with a big bag of rocks. He met a little girl named Sue. Sue looked at the rocks and said, "These rocks are very messy, but I can help you clean them." Bob was happy and thanked Sue. They cleaned the rocks together and made them shiny.
After cleaning the rocks, Bob an"""]
pat = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""" 
PAT_RE = regex.compile(pat)
def count_one(text):
    """对单个 corpus 元素做分词 + 词频统计，返回 Counter"""
    local = Counter() 
    paragraphs = regex.split('|'.join(map(regex.escape, special_tokens)), text) 
    for paragraph in paragraphs:
        for matches in PAT_RE.finditer(paragraph):
            word = matches.group(0)
            local[tuple(bytes([ch]) for ch in word.encode("utf-8"))] += 1
    return local 
def build_word_counts(corpus, num_process):
    with mp.Pool(num_process) as pool:
        # corpus 长度 == num_process，每个进程正好处理一个元素
        counters = pool.map(count_one, corpus)
        total = Counter()
    for c in counters:
        total.update(c)
    return total 

num_process = len(corpus)
word_counts = build_word_counts(corpus, num_process)
print(word_counts)

Counter({(b'.',): 56, (b' ', b't', b'h', b'e'): 42, (b',',): 38, (b' ', b'a', b'n', b'd'): 28, (b' ', b't', b'o'): 24, (b' ', b'w', b'a', b's'): 20, (b'\n',): 18, (b' ', b'a'): 18, (b' ', b'p', b'i', b't'): 18, (b' ', b'T', b'h', b'e', b'y'): 16, (b' ', b'b', b'a', b'l', b'l'): 14, (b' ', b'T', b'o', b'm'): 12, (b' ', b'b', b'u', b't'): 12, (b' ', b'B', b'o', b'b'): 12, (b' ', b'r', b'o', b'c', b'k', b's'): 12, (b' ', b'i', b'n'): 10, (b' ', b'b', b'i', b'g'): 10, (b' ', b'h', b'i', b's'): 8, (b' ', b's', b'c', b'a', b'r', b'e', b'd'): 6, (b' ', b'o', b'f'): 6, (b' ', b'm', b'o', b'l', b'e'): 6, (b' ', b'w', b'i', b't', b'h'): 6, (b' ', b'l', b'i', b't', b't', b'l', b'e'): 6, (b' ', b'v', b'e', b'r', b'y'): 6, (b' ', b'H', b'e'): 6, (b' ', b'n', b'a', b'm', b'e', b'd'): 6, (b' ', b'h', b'e', b'l', b'p'): 6, (b' ', b's', b'e', b'a', b'r', b'c', b'h'): 6, (b' ', b'f', b'o', b'r'): 6, (b' ', b'l', b'o', b'o', b'k', b'e', b'd'): 6, (b' ', b't', b'h', b'e', b'y'): 6, (b' ', b'c', b'o', b'u'

In [5]:
pair_counts: defaultdict[tuple[bytes], int] = defaultdict(int)    
pair2word: defaultdict[tuple[bytes], set[int]] = defaultdict(set) 
id2word:dict[int, tuple[bytes]] = dict()
for word_id, (word_bytes_tuple, counts) in enumerate(word_counts.items()):
        for i in range(len(word_bytes_tuple) - 1):
            pair1, pair2 = word_bytes_tuple[i], word_bytes_tuple[i + 1]
            pair_counts[(pair1, pair2)] += counts 
            pair2word[(pair1, pair2)].add(word_id) 
        id2word[word_id] = word_bytes_tuple 
print(f"pair counts is {pair_counts}") 
print(f"pair2word is {pair2word}")
print(f"id2word is {id2word}")

pair counts is defaultdict(<class 'int'>, {(b' ', b'd'): 16, (b'd', b'o'): 4, (b'o', b'n'): 10, (b"'", b't'): 2, (b' ', b'h'): 36, (b'h', b'a'): 8, (b'a', b'v'): 2, (b'v', b'e'): 12, (b' ', b't'): 106, (b't', b'o'): 40, (b' ', b'b'): 46, (b'b', b'e'): 6, (b' ', b's'): 46, (b's', b'c'): 8, (b'c', b'a'): 16, (b'a', b'r'): 26, (b'r', b'e'): 26, (b'e', b'd'): 46, (b' ', b'o'): 12, (b'o', b'f'): 6, (b't', b'h'): 70, (b'h', b'e'): 104, (b' ', b'l'): 24, (b'l', b'o'): 14, (b'o', b'u'): 18, (b'u', b'd'): 2, (b'o', b'g'): 4, (b' ', b'I'): 6, (b"'", b'l'): 2, (b'l', b'l'): 22, (b' ', b'p'): 26, (b'p', b'r'): 2, (b'r', b'o'): 14, (b'o', b't'): 8, (b't', b'e'): 8, (b'e', b'c'): 6, (b'c', b't'): 2, (b' ', b'y'): 4, (b'y', b'o'): 4, (b'"', b'.'): 2, (b' ', b'T'): 32, (b'T', b'h'): 24, (b' ', b'm'): 26, (b'm', b'o'): 6, (b'o', b'l'): 6, (b'l', b'e'): 24, (b' ', b'f'): 24, (b'f', b'e'): 8, (b'e', b'l'): 12, (b'l', b't'): 2, (b's', b'o'): 6, (b's', b'a'): 14, (b'a', b'f'): 4, (b' ', b'w'): 44, (b'w', b

In [6]:
def merge_word_with_best_pair(
        word_bytes_tuple : tuple[bytes],
        best_pair : tuple[bytes, bytes]
) -> tuple[bytes]:
    res = []
    i = 0
    while i < len(word_bytes_tuple):
        if i < len(word_bytes_tuple) - 1 and (best_pair == (word_bytes_tuple[i], word_bytes_tuple[i + 1])):
            res.append(best_pair[0] + best_pair[1])
            i += 2
        else:
            res.append(word_bytes_tuple[i])
            i += 1
    return tuple(res)  


In [ ]:
merges = [] 
vocab_size = 258 
cur_token_id = 257
while cur_token_id < vocab_size and len(pair_counts) > 0:
    best_pair = max(pair_counts, key=lambda item: (pair_counts[item], item))
    print(f"best_pair is{best_pair}")
    vocab[cur_token_id] = best_pair[0] + best_pair[1]
    merges.append(best_pair) 

    affected_word_ids = pair2word[best_pair] 
    print(f"affected_words are {[id2word[id] for id in affected_word_ids]}") 

    for id in list(affected_word_ids):
        word_bytes_tuple = id2word[id] 
        counts = word_counts[word_bytes_tuple]
        print(f"first affected word is {word_bytes_tuple} and counts is {counts}")
        for i in range(len(word_bytes_tuple) - 1):
            pair1, pair2 = word_bytes_tuple[i], word_bytes_tuple[i + 1]
            if  (pair1, pair2) in pair_counts:
                pair_counts[(pair1, pair2)] -= counts 
                pair2word[(pair1, pair2)].discard(id)
                if pair_counts[(pair1, pair2)] == 0:
                    del pair_counts[(pair1, pair2)] 
                    del pair2word[(pair1, pair2)]
        del word_counts[word_bytes_tuple] 
        print(f"pair_counts after remove old is {pair_counts}")
        print(f"pair2word after remove old is {pair2word}")
        word_merged = merge_word_with_best_pair(word_bytes_tuple, best_pair)
        word_counts[word_merged] += counts 
        id2word[id] = word_merged 
        for i in range(len(word_merged) - 1):
            pair1, pair2 = word_merged[i], word_merged[i + 1]
            pair_counts[(pair1, pair2)] += counts 
            pair2word[(pair1, pair2)].add(id)
        print(f"pair_counts after add new is {pair_counts}")
        print(f"pair2word after add new is {pair2word}") 
        print(f"word_counts is {word_counts}")
        print(f"id2word is{id2word}") 
        break
    cur_token_id += 1    


best_pair is(b' ', b't')
affected_words are [(b' ', b't', b'r', b'u', b's', b't'), (b' ', b't', b'o'), (b' ', b't', b'o', b'o'), (b' ', b't', b'h', b'e'), (b' ', b't', b'o', b'w', b'n'), (b' ', b't', b'r', b'i', b'e', b'd'), (b' ', b't', b'i', b'm', b'e'), (b' ', b't', b'h', b'e', b'y'), (b' ', b't', b'h', b'e', b'm'), (b' ', b't', b'h', b'e', b'r', b'e'), (b' ', b't', b'h', b'a', b'n', b'k', b'e', b'd'), (b' ', b't', b'h', b'i', b'n', b'k'), (b' ', b't', b'o', b'g', b'e', b't', b'h', b'e', b'r'), (b' ', b't', b'r', b'e', b'e', b's')]
first affected word is (b' ', b't', b'r', b'u', b's', b't') and counts is 2
pair_counts after remove old is defaultdict(<class 'int'>, {(b' ', b'd'): 16, (b'd', b'o'): 4, (b'o', b'n'): 10, (b"'", b't'): 2, (b' ', b'h'): 36, (b'h', b'a'): 8, (b'a', b'v'): 2, (b'v', b'e'): 12, (b' ', b't'): 104, (b't', b'o'): 40, (b' ', b'b'): 46, (b'b', b'e'): 6, (b' ', b's'): 46, (b's', b'c'): 8, (b'c', b'a'): 16, (b'a', b'r'): 26, (b'r', b'e'): 26, (b'e', b'd'): 46, (b' 